## TUGAS KECERDASAN ARTIFISIAL UNTUK GEOFISIKA
#### **Nama : Aurellia Kayla Putri Iskhas**
#### **NIM : H061261036**
---


# PRAKTIK RECIPE CONTOH

# 1. Recipe Audit CSV untuk QA/QC

## 1.1 Spesifikasi

**Masalah:** memeriksa struktur CSV, rentang waktu, nilai kosong, timestamp duplikat, dan kesesuaian unit tanpa mengubah file input.

**Input:**

- `CSV_PATH`: lokasi file CSV.
- `TIME_COLUMN`: nama kolom waktu.
- `UNITS`: kontrak unit untuk kolom numerik.
- `START_TIME`, `END_TIME`: batas waktu pemeriksaan.

**Output:**

- `REPORT_PATH`: laporan audit Markdown.
- `RUN_PATH`: catatan proses JSON.
- File CSV asli tidak diubah.

**Aturan:**

1. Kolom waktu wajib tersedia dan dapat diparsing.
2. Timestamp di luar rentang dilaporkan.
3. Nilai kosong dan timestamp duplikat dihitung.
4. Unit hanya dapat diverifikasi terhadap kontrak yang diberikan; nilai angka saja tidak membuktikan unit fisik.
5. Baris bermasalah dilaporkan, bukan dihapus otomatis.

## 1.2 Contoh input

Simpan sebagai `sample_audit.csv`:

```csv
timestamp,temperature,pressure
2026-01-01T00:00:00Z,25.2,1008
2026-01-01T01:00:00Z,25.5,1007
2026-01-01T01:00:00Z,,1007
2026-01-03T00:00:00Z,28.1,1005
```

Konfigurasi contoh:

- Kolom waktu: `timestamp`
- Unit: `temperature: degC`, `pressure: hPa`
- Rentang: `2026-01-01T00:00:00Z` sampai `2026-01-02T00:00:00Z`

## 1.3 Implementasi Python

Simpan sebagai `audit_csv.py`:

```python
import argparse
import json
from pathlib import Path

import pandas as pd


def audit_csv(
    csv_path,
    time_column,
    units,
    start_time,
    end_time,
    report_path,
    run_path,
):
    source = Path(csv_path)
    if not source.is_file():
        raise FileNotFoundError(source)

    df = pd.read_csv(source)

    if df.empty:
        raise ValueError("CSV tidak memiliki baris data.")

    if time_column not in df.columns:
        raise ValueError(
            f"Kolom waktu {time_column!r} tidak ditemukan."
        )

    # Pastikan batas waktu memiliki zona waktu yang konsisten.
    start = pd.Timestamp(start_time)
    end = pd.Timestamp(end_time)

    if start.tzinfo is None or end.tzinfo is None:
        raise ValueError(
            "START_TIME dan END_TIME wajib menyertakan zona waktu."
        )
    if start > end:
        raise ValueError("START_TIME harus <= END_TIME.")

    raw_time = df[time_column]
    parsed_time = pd.to_datetime(
        raw_time, errors="coerce", utc=True
    )

    start = start.tz_convert("UTC")
    end = end.tz_convert("UTC")

    missing = df.isna().sum().to_dict()
    invalid_time = int(parsed_time.isna().sum())

    valid_time = parsed_time.notna()
    outside_range = int(
        ((parsed_time < start) | (parsed_time > end))
        .where(valid_time, False)
        .sum()
    )

    duplicate_timestamp = int(
        parsed_time[valid_time].duplicated(keep=False).sum()
    )

    # Unit kontrak adalah metadata deklaratif, bukan deteksi unit.
    unit_check = {}
    for column, unit in units.items():
        unit_check[column] = {
            "column_exists": column in df.columns,
            "declared_unit": unit,
        }

    report = {
        "input": str(source.resolve()),
        "rows": int(len(df)),
        "columns": list(df.columns),
        "time_column": time_column,
        "time_range_utc": {
            "start": start.isoformat(),
            "end": end.isoformat(),
        },
        "missing_values_by_column": {
            key: int(value) for key, value in missing.items()
        },
        "invalid_timestamps": invalid_time,
        "timestamps_outside_range": outside_range,
        "rows_with_duplicate_timestamps": duplicate_timestamp,
        "unit_contract": unit_check,
        "input_modified": False,
    }

    report_file = Path(report_path)
    run_file = Path(run_path)
    report_file.parent.mkdir(parents=True, exist_ok=True)
    run_file.parent.mkdir(parents=True, exist_ok=True)

    lines = [
        "# Laporan Audit CSV",
        "",
        f"- Input: `{source}`",
        f"- Jumlah baris: {len(df)}",
        f"- Kolom waktu: `{time_column}`",
        f"- Timestamp tidak valid: {invalid_time}",
        f"- Timestamp di luar rentang: {outside_range}",
        f"- Baris dengan timestamp duplikat: "
        f"{duplicate_timestamp}",
        "",
        "## Nilai kosong",
    ]

    for column, count in missing.items():
        lines.append(f"- `{column}`: {count}")

    lines += ["", "## Kontrak unit"]
    for column, item in unit_check.items():
        lines.append(
            f"- `{column}`: {item['declared_unit']} "
            f"(kolom tersedia: {item['column_exists']})"
        )

    report_file.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )

    run_file.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--time-column", required=True)
    parser.add_argument("--start", required=True)
    parser.add_argument("--end", required=True)
    parser.add_argument("--report", default="output/audit.md")
    parser.add_argument("--run", default="output/run.json")
    parser.add_argument(
        "--units",
        default="{}",
        help='JSON, contoh: {"temperature":"degC"}',
    )
    args = parser.parse_args()

    result = audit_csv(
        csv_path=args.input,
        time_column=args.time_column,
        units=json.loads(args.units),
        start_time=args.start,
        end_time=args.end,
        report_path=args.report,
        run_path=args.run,
    )

    print(json.dumps(result, indent=2, ensure_ascii=False))
```

## 1.4 Instalasi dan menjalankan

Instal dependensi:

```bash
python -m pip install pandas
```

Jalankan program:

```bash
python audit_csv.py \
  --input sample_audit.csv \
  --time-column timestamp \
  --start 2026-01-01T00:00:00Z \
  --end 2026-01-02T00:00:00Z \
  --units '{"temperature":"degC","pressure":"hPa"}' \
  --report output/audit.md \
  --run output/run.json
```

Di Windows, perintah dapat ditulis dalam satu baris atau menggunakan karakter kelanjutan baris yang sesuai dengan terminal.

## 1.5 Pemeriksaan hasil yang diharapkan

Untuk contoh input di atas:

- Jumlah baris: 4.
- Nilai kosong: 1 pada `temperature`.
- Timestamp duplikat: 2 baris terlibat dalam timestamp yang berulang.
- Timestamp di luar rentang: 1.
- Timestamp tidak valid: 0.
- File input tetap sama.

**Kriteria penerimaan:**

- Laporan Markdown dan JSON terbentuk.
- Nilai hitungan cocok dengan contoh input.
- Kolom yang tidak tercantum dalam kontrak unit tidak otomatis dianggap salah.
- Input tidak ditulis ulang.

**Keterbatasan:** recipe ini belum memverifikasi satuan dari sumber instrumen, batas fisik nilai sensor, urutan waktu, atau frekuensi sampling yang seharusnya. Pemeriksaan tersebut memerlukan kontrak data tambahan.

---

# 2. Recipe Normalisasi GeoJSON

## 2.1 Spesifikasi

**Masalah:** menghasilkan GeoJSON ternormalisasi, inventaris CSV, dan ringkasan JSON tanpa kehilangan Feature ID, geometri, properti, geometri null, atau urutan fitur.

**Input:**

- `GEOJSON_PATH`: file GeoJSON sumber.
- `OUTPUT_DIR`: direktori hasil.
- CRS dan kontrak sumbu ditentukan berdasarkan metadata sumber atau dikonfirmasi kepada pemilik data.

**Output:**

- `normalized.geojson`
- `inventory.csv`
- `summary.json`

**Prinsip:**

1. Validasi struktur FeatureCollection.
2. Pertahankan urutan fitur dan properti.
3. Pertahankan ID yang tersedia.
4. Jangan mengisi geometri null dengan geometri buatan.
5. Jangan mengubah CRS atau urutan koordinat tanpa bukti kontrak.
6. Jangan mengklaim validasi topologi lengkap jika hanya pemeriksaan struktur yang dilakukan.

## 2.2 Contoh input

Simpan sebagai `sample.geojson`:

```json
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "id": "A01",
      "properties": {
        "name": "Lokasi A",
        "class": "station"
      },
      "geometry": {
        "type": "Point",
        "coordinates": [119.4, -5.1]
      }
    },
    {
      "type": "Feature",
      "id": "A02",
      "properties": {
        "name": "Lokasi B",
        "class": "station"
      },
      "geometry": null
    }
  ]
}
```

Contoh ini menggunakan koordinat geografis yang diasumsikan mengikuti urutan longitude, latitude. Asumsi tersebut harus dikonfirmasi dari sumber sebelum digunakan untuk data produksi.

## 2.3 Implementasi Python

Simpan sebagai `normalize_geojson.py`:

```python
import argparse
import csv
import json
from collections import Counter
from pathlib import Path


def normalize_geojson(input_path, output_dir):
    input_file = Path(input_path)
    out = Path(output_dir)

    if not input_file.is_file():
        raise FileNotFoundError(input_file)

    out.mkdir(parents=True, exist_ok=True)

    data = json.loads(input_file.read_text(encoding="utf-8"))

    if data.get("type") != "FeatureCollection":
        raise ValueError("Root harus bertipe FeatureCollection.")

    features = data.get("features")
    if not isinstance(features, list):
        raise ValueError("'features' harus berupa array.")

    geometry_types = Counter()
    null_geometry = 0
    missing_ids = 0
    invalid_features = []

    # Salin struktur tingkat atas dan fitur tanpa membangun ulang
    # objek geometri atau properti.
    normalized = dict(data)
    normalized["features"] = []

    inventory = []

    for index, feature in enumerate(features):
        if not isinstance(feature, dict):
            invalid_features.append(index)
            continue

        if feature.get("type") != "Feature":
            invalid_features.append(index)
            continue

        geometry = feature.get("geometry")
        properties = feature.get("properties")
        feature_id = feature.get("id")

        if feature_id is None:
            missing_ids += 1

        if geometry is None:
            geom_type = "NULL"
            null_geometry += 1
        elif isinstance(geometry, dict):
            geom_type = geometry.get("type", "UNKNOWN")
            geometry_types[geom_type] += 1
        else:
            geom_type = "INVALID"
            invalid_features.append(index)

        if properties is None:
            properties = {}

        if not isinstance(properties, dict):
            invalid_features.append(index)
            continue

        # Fitur valid dipertahankan dengan urutan asli.
        normalized["features"].append(feature)

        inventory.append({
            "index": index,
            "feature_id": feature_id,
            "geometry_type": geom_type,
            "properties": json.dumps(
                properties, ensure_ascii=False
            ),
        })

    geojson_path = out / "normalized.geojson"
    inventory_path = out / "inventory.csv"
    summary_path = out / "summary.json"

    geojson_path.write_text(
        json.dumps(normalized, ensure_ascii=False, indent=2)
        + "\n",
        encoding="utf-8",
    )

    with inventory_path.open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "index",
                "feature_id",
                "geometry_type",
                "properties",
            ],
        )
        writer.writeheader()
        writer.writerows(inventory)

    summary = {
        "input": str(input_file),
        "input_feature_count": len(features),
        "output_feature_count": len(normalized["features"]),
        "geometry_type_counts": dict(geometry_types),
        "null_geometry_count": null_geometry,
        "missing_id_count": missing_ids,
        "invalid_feature_indices": invalid_features,
        "feature_order_preserved": True,
        "crs_transformed": False,
        "topology_validation": "not_performed",
        "network_access": False,
    }

    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    if invalid_features:
        raise ValueError(
            "Ditemukan fitur tidak valid; periksa summary.json. "
            "Output yang terbentuk hanya mencakup fitur yang diterima."
        )

    return summary


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output-dir", default="output_geojson")
    args = parser.parse_args()

    result = normalize_geojson(args.input, args.output_dir)
    print(json.dumps(result, indent=2, ensure_ascii=False))
```

## 2.4 Menjalankan

```bash
python normalize_geojson.py \
  --input sample.geojson \
  --output-dir output_geojson
```

Tidak diperlukan pustaka GIS eksternal untuk pemeriksaan struktur dasar ini.

## 2.5 Pemeriksaan hasil

Untuk contoh input:

- Input Feature: 2.
- Output Feature: 2.
- Geometri null: 1.
- ID hilang: 0.
- Transformasi CRS: tidak dilakukan.
- Akses jaringan: tidak dilakukan.

Pastikan:

- `normalized.geojson` dapat dibaca kembali sebagai JSON.
- ID `A01` dan `A02` tetap ada.
- Geometri null tetap null.
- Properti setiap fitur tidak hilang.
- Urutan fitur sama dengan sumber.

**Batasan penting:** ini merupakan normalisasi struktur dasar, bukan validasi geometri atau topologi penuh. Fitur yang ditolak dapat menyebabkan output tidak lengkap; dalam alur produksi, kegagalan validasi sebaiknya membatalkan publikasi output final. Jangan menetapkan CRS berdasarkan perkiraan koordinat semata.

---

# 3. Recipe Simulasi Difusi Panas Satu Dimensi

## 3.1 Spesifikasi

**Masalah:** menyimulasikan difusi panas satu dimensi dengan profil awal, difusivitas termal, kondisi batas, grid ruang, dan langkah waktu.

Persamaan pengatur:

$$
\frac{\partial T}{\partial t}
=
\alpha\frac{\partial^2T}{\partial x^2}
$$

Keterangan:

- `T`: temperatur, °C.
- `t`: waktu, s.
- `x`: posisi, m.
- $\alpha$: difusivitas termal, m²/s.

Diskretisasi eksplisit:

$$
T_i^{n+1}
=
T_i^n+
r(T_{i+1}^n-2T_i^n+T_{i-1}^n)
$$

$$
r=\frac{\alpha\Delta t}{\Delta x^2}
$$

Untuk skema eksplisit 1D dengan difusivitas positif dan grid seragam, syarat kestabilan dasar adalah:

$$
0 < r \leq \frac{1}{2}
$$

Syarat tersebut bukan bukti bahwa model fisik sudah terkalibrasi.

## 3.2 Konfigurasi contoh

- Panjang domain: 1 m.
- Temperatur awal: 100 °C di tengah, 20 °C di tempat lain.
- Difusivitas: $1.0\times10^{-4}$ m²/s.
- Batas kiri dan kanan: temperatur tetap 20 °C.
- Jumlah interval ruang: 50.
- Langkah waktu: 0,1 s.
- Durasi simulasi: 10 s.

## 3.3 Implementasi Python

Simpan sebagai `heat_diffusion_1d.py`:

```python
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def simulate(alpha, length, nx, dt, duration, left, right, output):
    if alpha <= 0 or length <= 0:
        raise ValueError("alpha dan length harus positif.")
    if nx < 2 or dt <= 0 or duration <= 0:
        raise ValueError("Grid dan waktu tidak valid.")

    dx = length / nx
    r = alpha * dt / dx**2

    if r > 0.5:
        raise ValueError(
            f"Skema eksplisit tidak stabil: r={r:.6f} > 0.5. "
            "Kurangi dt atau perbesar dx."
        )

    x = np.linspace(0, length, nx + 1)
    T = np.full(nx + 1, 20.0)
    T[np.argmin(abs(x - length / 2))] = 100.0
    T[0], T[-1] = left, right

    nsteps = int(np.ceil(duration / dt))
    actual_dt = duration / nsteps
    r = alpha * actual_dt / dx**2

    if r > 0.5:
        raise ValueError("Langkah waktu aktual melanggar kestabilan.")

    history = [T.copy()]
    times = [0.0]

    for step in range(1, nsteps + 1):
        new_T = T.copy()
        new_T[1:-1] = T[1:-1] + r * (
            T[2:] - 2 * T[1:-1] + T[:-2]
        )
        new_T[0], new_T[-1] = left, right
        T = new_T

        if step == nsteps or step % max(1, nsteps // 20) == 0:
            history.append(T.copy())
            times.append(step * actual_dt)

    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)

    np.savetxt(
        output / "temperature_profile.csv",
        np.column_stack([x, T]),
        delimiter=",",
        header="x_m,temperature_degC",
        comments="",
    )

    plt.figure()
    for index in np.linspace(
        0, len(history) - 1, min(5, len(history)), dtype=int
    ):
        plt.plot(x, history[index], label=f"t={times[index]:.2f}s")
    plt.xlabel("Posisi (m)")
    plt.ylabel("Temperatur (°C)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output / "temperature_profile.png", dpi=150)
    plt.close()

    run = {
        "alpha_m2_s": alpha,
        "length_m": length,
        "nx_intervals": nx,
        "dx_m": dx,
        "dt_requested_s": dt,
        "dt_actual_s": actual_dt,
        "duration_s": duration,
        "steps": nsteps,
        "stability_ratio": r,
        "stability_check": "passed" if r <= 0.5 else "failed",
        "boundary_conditions": {
            "left_degC": left,
            "right_degC": right,
        },
        "status": "completed_by_script",
        "calibration": "not_performed",
    }

    (output / "run.json").write_text(
        json.dumps(run, indent=2), encoding="utf-8"
    )
    return run


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--alpha", type=float, default=1e-4)
    parser.add_argument("--length", type=float, default=1.0)
    parser.add_argument("--nx", type=int, default=50)
    parser.add_argument("--dt", type=float, default=0.1)
    parser.add_argument("--duration", type=float, default=10.0)
    parser.add_argument("--left", type=float, default=20.0)
    parser.add_argument("--right", type=float, default=20.0)
    parser.add_argument("--output", default="heat_output")
    args = parser.parse_args()

    result = simulate(
        args.alpha, args.length, args.nx,
        args.dt, args.duration,
        args.left, args.right, args.output,
    )
    print(json.dumps(result, indent=2))
```

## 3.4 Menjalankan

Instal dependensi:

```bash
python -m pip install numpy matplotlib
```

Jalankan program:

```bash
python heat_diffusion_1d.py \
  --alpha 0.0001 \
  --length 1 \
  --nx 50 \
  --dt 0.1 \
  --duration 10 \
  --output heat_output
```

## 3.5 Pemeriksaan hasil

Dengan parameter tersebut:

- $\Delta x = 1/50 = 0.02$ m.
- $r = 0.0001(0.1)/(0.02)^2 = 0.025$.
- Nilai `r` memenuhi syarat kestabilan eksplisit.
- Output: `temperature_profile.csv`, `temperature_profile.png`, dan `run.json`.

**Pemeriksaan tambahan:**

1. Temperatur tidak boleh menghasilkan nilai non-finite.
2. Temperatur pada kedua batas tetap sesuai kondisi batas.
3. Pada konfigurasi ini, temperatur puncak diharapkan menyebar dan menurun terhadap waktu.
4. Ulangi dengan grid lebih halus dan langkah waktu yang sesuai untuk menguji sensitivitas numerik.

**Batasan:** simulasi ini mengasumsikan material homogen, difusivitas konstan, dan tidak ada sumber panas internal. Belum ada validasi eksperimental, kalibrasi, atau pembuktian konvergensi.

---

# Checklist akhir

- [ ] Ketiga skrip disimpan.
- [ ] Dependensi terpasang.
- [ ] Contoh input dibuat.
- [ ] Ketiga perintah dijalankan.
- [ ] Output diperiksa secara independen.
- [ ] Status eksekusi dicatat berdasarkan hasil aktual.
- [ ] File sumber tidak tertimpa oleh proses.



---

# RECIPE QA/QC Curah Hujan, Inventarisasi Raster DEM, dan Analisis Spektrum Seismik

# 1. Recipe Deteksi Outlier Curah Hujan

## 1.1 Tujuan dan masalah

Mendeteksi nilai curah hujan yang berpotensi anomali tanpa langsung menghapus data asli.

Recipe memeriksa:

- Struktur kolom dan tipe data.
- Timestamp duplikat atau tidak valid.
- Nilai curah hujan negatif dan nilai kosong.
- Outlier statistik berdasarkan jendela waktu.
- Perbedaan antara anomali statistik dan kesalahan pengukuran yang telah dikonfirmasi.

Nilai ekstrem tidak selalu salah. Hujan sangat lebat dapat merupakan kejadian meteorologis yang valid.

## 1.2 Kontrak input

- `RAIN_CSV_PATH`: file CSV.
- `TIME_COLUMN`: nama kolom timestamp.
- `RAIN_COLUMN`: nama kolom curah hujan.
- `RAIN_UNIT`: satuan, misalnya mm per interval.
- `INTERVAL_MINUTES`: durasi interval pengukuran.
- `METHOD`: `rolling_mad`, `iqr`, atau metode lain yang dinyatakan eksplisit.
- `WINDOW_SIZE`: jumlah observasi dalam jendela.
- `THRESHOLD`: ambang deteksi.
- `OUTPUT_DIR`: direktori hasil.

Kolom waktu wajib memiliki zona waktu yang diketahui. Unit harus menyatakan apakah curah hujan berupa akumulasi per interval, intensitas mm/jam, atau akumulasi harian.

## 1.3 Rancangan algoritme

1. Baca CSV dan validasi skema.
2. Parse timestamp tanpa mengubah data sumber.
3. Periksa timestamp berulang, urutan waktu, dan interval yang hilang.
4. Validasi nilai curah hujan negatif, kosong, dan nonnumerik.
5. Hitung skor anomali dengan metode yang dikonfigurasi.
6. Simpan flag anomali, skor, dan alasan flag per observasi.
7. Buat ringkasan statistik serta catatan konfigurasi.
8. Jangan menghapus atau mengganti nilai asli secara otomatis.


Untuk metode Median Absolute Deviation (MAD), rumusnya adalah:

$$
MAD = \operatorname{median}\left(|x_i-\tilde{x}|\right)
$$

Skor anomali dihitung menggunakan persamaan:

$$
z_i^* = \frac{0.6745(x_i-\tilde{x})}{MAD}
$$


Dengan x-tilde (median jendela) sebagai nilai median. Observasi dapat ditandai ketika nilai absolut skor anomali memenuhi:

$$
|z_i^*| > \text{threshold}
$$

dengan `threshold` merupakan nilai ambang yang telah dikonfigurasi.

Jika MAD nol, skor tidak boleh dihitung dengan pembagian nol. Gunakan status khusus atau fallback yang terdokumentasi. Jendela *rolling* juga perlu menetapkan ukuran minimum observasi dan perlakuan nilai kosong.

## 1.4 Contoh input

Simpan sebagai CSV:

```csv
timestamp,rain_mm
2026-01-01T00:00:00Z,0.0
2026-01-01T01:00:00Z,0.2
2026-01-01T02:00:00Z,0.0
2026-01-01T03:00:00Z,45.0
2026-01-01T04:00:00Z,0.1
2026-01-01T05:00:00Z,-1.0
```

Baris 45,0 mm berpotensi ekstrem dan perlu diperiksa terhadap durasi interval serta kondisi setempat. Baris -1,0 mm melanggar batas fisik curah hujan nonnegatif apabila kolom memang menyatakan akumulasi hujan.

## 1.5 Rencana implementasi Python

Gunakan `pandas` dan `numpy`. Modul inti yang direkomendasikan:

```python
import numpy as np
import pandas as pd


def rolling_mad_flags(series, window=7, threshold=3.5, min_periods=5):
    """Contoh fungsi statistik; validasi konfigurasi sebelum digunakan."""
    median = series.rolling(
        window, center=True, min_periods=min_periods
    ).median()

    deviation = (series - median).abs()
    mad = deviation.rolling(
        window, center=True, min_periods=min_periods
    ).median()

    score = 0.6745 * (series - median) / mad.replace(0, np.nan)
    flag = score.abs() > threshold
    return score, flag
```

**Catatan implementasi:** ekspresi default `window` telah ditulis sebagai `window=7` agar kode valid secara sintaks. Kode produksi tetap perlu melalui pemeriksaan konfigurasi dan pengujian unit.

Alur produksi perlu memisahkan flag berikut:

- `invalid_value`: nilai negatif atau nonnumerik.
- `statistical_outlier`: kandidat anomali statistik.
- `duplicate_timestamp`: waktu berulang.
- `missing_interval`: interval observasi yang hilang.
- `review_status`: belum diperiksa, dikonfirmasi, atau dinyatakan valid.

## 1.6 Target output

- `rainfall_flags.csv`: timestamp, nilai asli, skor, flag, dan alasan.
- `rainfall_summary.json`: jumlah observasi, nilai kosong, jumlah flag, rentang waktu, parameter, dan unit.
- `rainfall_timeseries.png`: grafik deret waktu dengan penanda kandidat anomali.
- `run.json`: konfigurasi, versi perangkat lunak, status proses, dan pesan peringatan.

## 1.7 Pemeriksaan dan kriteria penerimaan

- Nilai negatif ditandai sebagai pelanggaran validasi.
- Data asli tetap dapat dibandingkan dengan input.
- Flag statistik tidak otomatis mengubah atau menghapus data.
- Ambang, jendela, unit, dan zona waktu tercatat dalam output.
- Kasus MAD nol dan data yang terlalu sedikit ditangani.
- Hasil diuji pada seri konstan, data kosong, duplikat timestamp, dan nilai ekstrem buatan.
- Outlier yang ditemukan tidak dinyatakan sebagai kesalahan sensor tanpa bukti tambahan.

**Batasan:** deteksi statistik tidak menggantikan pemeriksaan meteorologis, metadata alat ukur, atau analisis kejadian hujan ekstrem.

---

# 2. Recipe Inventarisasi Raster DEM

## 2.1 Tujuan dan masalah

Menginventarisasi sekumpulan raster *Digital Elevation Model* (DEM) untuk mengetahui cakupan spasial, resolusi, sistem referensi koordinat, karakteristik nilai elevasi, nodata, dan kelayakan pemrosesan berikutnya.

Recipe ini tidak mengubah raster sumber dan tidak melakukan reproyeksi otomatis.

## 2.2 Kontrak input

- `DEM_PATH`: satu file raster atau direktori berisi raster.
- `RECURSIVE`: apakah subdirektori diperiksa.
- `OUTPUT_DIR`: lokasi hasil.
- Format yang didukung: raster yang dapat dibaca oleh GDAL/rasterio.
- CRS, satuan elevasi, dan datum vertikal harus dicatat apabila tersedia.

Input dapat berupa GeoTIFF atau format raster lain yang didukung lingkungan GDAL.

## 2.3 Rancangan algoritme

1. Temukan file raster sesuai aturan pencarian.
2. Baca metadata tanpa mengubah file.
3. Catat dimensi raster, jumlah band, tipe data, transformasi affine, dan CRS.
4. Catat resolusi piksel dan batas cakupan.
5. Periksa nodata dan nilai valid.
6. Hitung statistik elevasi dari piksel valid.
7. Tandai CRS tidak tersedia, transformasi affine bermasalah, atau cakupan yang tidak masuk akal.
8. Ekspor inventaris dan ringkasan.

Statistik elevasi harus memperhitungkan nodata dan mask raster. Nilai ekstrem dapat menunjukkan kesalahan, tetapi juga dapat merupakan elevasi yang valid atau kesalahan interpretasi unit.

## 2.4 Contoh struktur input

```text
dem_data/
├── dem_tile_001.tif
├── dem_tile_002.tif
└── dem_tile_003.tif
```

Tidak ada asumsi bahwa semua tile menggunakan CRS, resolusi, datum vertikal, atau satuan yang sama.

## 2.5 Rencana implementasi Python

Dependensi yang direkomendasikan:

```bash
python -m pip install rasterio numpy pandas
```

Kerangka fungsi inventaris:

```python
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio


def inspect_dem(path):
    path = Path(path)

    with rasterio.open(path) as ds:
        row = {
            "file": str(path),
            "width": ds.width,
            "height": ds.height,
            "band_count": ds.count,
            "crs": str(ds.crs) if ds.crs else None,
            "nodata": ds.nodata,
            "dtype": ",".join(ds.dtypes),
            "pixel_size_x": ds.transform.a,
            "pixel_size_y": ds.transform.e,
            "bounds_left": ds.bounds.left,
            "bounds_bottom": ds.bounds.bottom,
            "bounds_right": ds.bounds.right,
            "bounds_top": ds.bounds.top,
        }

        # Statistik pada band pertama; masked array mengecualikan
        # piksel yang dimask oleh raster.
        band = ds.read(1, masked=True)
        valid = band.compressed()
        row["valid_pixel_count"] = int(valid.size)

        if valid.size:
            row["elevation_min"] = float(np.min(valid))
            row["elevation_max"] = float(np.max(valid))
            row["elevation_mean"] = float(np.mean(valid))
            row["elevation_std"] = float(np.std(valid))
        else:
            row["elevation_min"] = None
            row["elevation_max"] = None
            row["elevation_mean"] = None
            row["elevation_std"] = None

        return row


def inventory_dem(directory, output_csv):
    root = Path(directory)
    files = sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    )

    records = []
    errors = []

    for path in files:
        try:
            records.append(inspect_dem(path))
        except Exception as exc:
            errors.append({
                "file": str(path),
                "error": str(exc),
            })

    pd.DataFrame(records).to_csv(output_csv, index=False)
    return records, errors
```

Kerangka ini perlu dilengkapi dengan CLI, ringkasan JSON, validasi direktori, penanganan band, dan pengujian sebelum dijadikan pipeline produksi. Jika raster memiliki mask internal, mask tersebut perlu diperiksa untuk memastikan interpretasi nodata sesuai sumber.

## 2.6 Target output

- `dem_inventory.csv`: satu baris per file raster.
- `dem_summary.json`: jumlah file, jumlah sukses/gagal, CRS yang ditemukan, rentang resolusi, serta peringatan.
- `dem_errors.csv`: file yang gagal dibaca dan pesan kesalahan.
- `run.json`: versi perangkat lunak, parameter, waktu proses, dan status.

## 2.7 Pemeriksaan dan kriteria penerimaan

- Semua file yang ditemukan tercatat sebagai berhasil atau gagal.
- Dimensi dan CRS cocok dengan metadata raster.
- Statistik hanya menggunakan piksel valid.
- Resolusi dan cakupan ditampilkan dalam satuan CRS yang benar.
- DEM dengan CRS berbeda tidak digabungkan diam-diam.
- Unit elevasi dan datum vertikal tidak ditebak dari angka saja.
- File sumber tidak dimodifikasi.

**Batasan:** inventarisasi tidak membuktikan akurasi vertikal DEM, kualitas permukaan, kesesuaian datum vertikal, atau keselarasan antar-tile. Pemeriksaan overlap, seam, dan kualitas elevasi memerlukan tahap QA tambahan.

---

# 3. Recipe Analisis Spektrum Sinyal Seismik

## 3.1 Tujuan dan masalah

Menghasilkan spektrum amplitudo dan estimasi frekuensi dominan dari sinyal seismik dengan memeriksa sampling, nilai hilang, tren, dan kemungkinan aliasing.

Recipe ini ditujukan untuk analisis sinyal, bukan penentuan magnitudo gempa atau interpretasi sumber seismik secara otomatis.

## 3.2 Kontrak input

- `SIGNAL_PATH`: file CSV.
- `TIME_COLUMN`: timestamp atau waktu relatif.
- `SIGNAL_COLUMN`: komponen sinyal yang dianalisis.
- `SAMPLING_RATE_HZ`: frekuensi sampling jika tidak dapat ditentukan dari timestamp.
- `UNIT`: satuan sinyal, misalnya m/s² atau count instrumen.
- `WINDOW`: jenis window, misalnya Hann.
- `DETREND`: aturan penghilangan tren.
- `OUTPUT_DIR`: direktori hasil.

Sebelum analisis, pastikan data memiliki interval sampling yang sesuai dan tidak ada gap yang diabaikan.

## 3.3 Dasar teori


Untuk sinyal diskret $x[n]$, *Discrete Fourier Transform* (DFT) didefinisikan sebagai:

$$
X[k] = \sum_{n=0}^{N-1} x[n]e^{-j2\pi kn/N}
$$

Resolusi frekuensi dasar:

$$
\Delta f = \frac{f_s}{N}
$$

Frekuensi Nyquist:

$$
f_N = \frac{f_s}{2}
$$

Dengan:

- $f_s$: frekuensi sampling dalam Hz.
- $N$: jumlah sampel.
- $\Delta f$: jarak antarbin frekuensi.


Spektrum amplitudo bergantung pada normalisasi, window, panjang sinyal, dan konvensi satu sisi atau dua sisi. Karena itu, satuan dan metode normalisasi harus dicatat bersama hasil.

## 3.4 Rancangan algoritme

1. Baca data dan validasi kolom.
2. Tentukan frekuensi sampling dari metadata atau timestamp.
3. Deteksi gap, duplikasi waktu, dan interval tidak seragam.
4. Konversikan sinyal menjadi numerik dan periksa nilai non-finite.
5. Hapus tren sesuai konfigurasi.
6. Terapkan window yang dinyatakan.
7. Hitung FFT satu sisi dan spektrum amplitudo dengan normalisasi yang terdokumentasi.
8. Hitung frekuensi dominan dalam rentang analisis yang dikonfigurasi.
9. Simpan spektrum, plot, dan catatan run.
10. Berikan peringatan jika sinyal terlalu pendek atau sampling tidak memadai.

## 3.5 Contoh input

Simpan sebagai `seismic_signal.csv`:

```csv
time_s,acceleration_m_s2
0.00,0.02
0.01,0.05
0.02,0.01
0.03,-0.04
0.04,-0.02
0.05,0.03
```

Data ini hanya ilustrasi struktur. Enam sampel tidak memadai untuk analisis spektrum seismik yang andal.

## 3.6 Rencana implementasi Python

Dependensi:

```bash
python -m pip install numpy pandas scipy matplotlib
```

Contoh fungsi analisis spektrum:

```python
import numpy as np
from scipy.signal import detrend, windows


def amplitude_spectrum(signal, sampling_rate_hz):
    x = np.asarray(signal, dtype=float)

    if x.ndim != 1 or x.size < 16:
        raise ValueError("Sinyal harus 1D dengan minimal 16 sampel.")
    if not np.all(np.isfinite(x)):
        raise ValueError("Sinyal mengandung NaN atau nilai non-finite.")
    if sampling_rate_hz <= 0:
        raise ValueError("Sampling rate harus positif.")

    x = detrend(x, type="linear")
    n = x.size

    window = windows.hann(n, sym=False)
    xw = x * window

    spectrum = np.fft.rfft(xw)
    frequency = np.fft.rfftfreq(n, d=1.0 / sampling_rate_hz)

    # Normalisasi amplitudo untuk spektrum satu sisi.
    amplitude = np.abs(spectrum) / window.sum()

    if n > 1:
        if n % 2 == 0:
            amplitude[1:-1] *= 2
        else:
            amplitude[1:] *= 2

    dominant_index = int(np.argmax(amplitude[1:]) + 1)

    return {
        "frequency_hz": frequency,
        "amplitude": amplitude,
        "dominant_frequency_hz": float(
            frequency[dominant_index]
        ),
        "frequency_resolution_hz": sampling_rate_hz / n,
        "nyquist_hz": sampling_rate_hz / 2,
        "sample_count": n,
        "window": "hann",
        "detrend": "linear",
    }
```

Frekuensi dominan pada contoh fungsi dihitung tanpa bin DC. Untuk analisis produksi, tentukan rentang frekuensi yang relevan agar noise frekuensi tinggi atau artefak tidak langsung dianggap sebagai frekuensi dominan seismik.

## 3.7 Target output

- `spectrum.csv`: frekuensi (Hz) dan amplitudo.
- `spectrum.png`: grafik spektrum.
- `summary.json`: frekuensi dominan, resolusi frekuensi, Nyquist, jumlah sampel, sampling rate, unit, dan parameter analisis.
- `run.json`: konfigurasi dan status validasi.

## 3.8 Pemeriksaan dan kriteria penerimaan

- Sampling rate positif dan didukung metadata atau timestamp.
- Interval sampling diperiksa untuk mendeteksi gap dan jitter.
- Nilai NaN atau infinite tidak diproses diam-diam.
- Frekuensi hasil tidak melebihi frekuensi Nyquist.
- Resolusi frekuensi konsisten dengan jumlah sampel dan sampling rate.
- Normalisasi amplitudo dan jenis window tercatat.
- Frekuensi dominan dibandingkan dengan sinyal sintetis yang frekuensinya diketahui.
- Output tidak digunakan untuk klaim magnitudo, lokasi sumber, atau mekanisme gempa tanpa metode dan bukti tambahan.

**Batasan:** spektrum amplitudo tidak otomatis sama dengan *power spectral density*. Analisis instrumen juga memerlukan respons sensor, kalibrasi, satuan, serta koreksi instrumen bila hasil fisik absolut diperlukan.

---

# Checklist akhir 

- [ ] Kontrak input dan output disepakati.
- [ ] Unit, CRS, zona waktu, atau sampling rate ditetapkan dari metadata.
- [ ] Validasi input dilakukan sebelum analisis.
- [ ] Kasus tepi diuji.
- [ ] Data sumber tidak diubah secara diam-diam.
- [ ] Peringatan dan kegagalan dicatat.
- [ ] Status eksekusi hanya ditetapkan setelah proses nyata dijalankan.
- [ ] Hasil ilmiah tidak dinyatakan valid hanya karena kode selesai berjalan.
